In [1]:
%pip install scikit-learn matplotlib numpy opencv-python filterpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Pipeline d'Analyse de Trajectoire de Drone

Ce notebook consolide l'ensemble du projet "Projet AI et TI" en un seul document interactif.
Il permet d'analyser le comportement des drones (Normal, Suspect, Hostile) à partir de données de tracking (CSV) et d'une vidéo.

## Fonctionnalités
1. **Lissage de Trajectoire**: Utilise un Filtre de Kalman pour réduire le bruit.
2. **Extraction de Caractéristiques**: Calcule la vitesse, l'accélération et les virages.
3. **Prédiction de Trajectoire**: Utilise la régression polynomiale pour prédire les mouvements futurs.
4. **Analyse de Comportement**: Système expert detectant les comportements hostiles.
5. **Visualisation**: Génère une vidéo annotée avec des alertes visuelles et sonores.

## 1. Importation des Librairies

In [2]:
import cv2
import json
import csv
import numpy as np
import wave
import struct
import subprocess
import os
import winsound
import threading
from collections import defaultdict
from collections import deque
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configuration pour l'affichage dans le notebook
%matplotlib inline

## 2. Module: Filtre de Kalman

In [3]:
"""
Module de lissage et prédiction de trajectoire avec filtre de Kalman
"""
import numpy as np
from filterpy.kalman import KalmanFilter


class DroneKalmanFilter:
    """
    Filtre de Kalman pour lisser et prédire la trajectoire d'un drone.
    État: [x, y, vx, vy] (position et vitesse)
    """
    
    def __init__(self):
        """Initialise le filtre de Kalman 2D"""
        self.kf = KalmanFilter(dim_x=4, dim_z=2)
        
        # Matrice de transition d'état (modèle de mouvement)
        # x(k+1) = x(k) + vx(k)*dt
        # y(k+1) = y(k) + vy(k)*dt
        dt = 1.0  # timestep (1 frame)
        self.kf.F = np.array([
            [1, 0, dt, 0],
            [0, 1, 0, dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ])
        
        # Matrice d'observation (on observe uniquement x, y)
        self.kf.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ])
        
        # Covariance du bruit de mesure
        self.kf.R *= 10  # Bruit sur les observations
        
        # Covariance du bruit de processus
        self.kf.Q *= 0.1  # Incertitude du modèle
        
        # Covariance de l'état initial
        self.kf.P *= 100
        
        self.initialized = False
    
    def update(self, x, y):
        """
        Met à jour le filtre avec une nouvelle observation
        
        Args:
            x (float): coordonnée x observée
            y (float): coordonnée y observée
            
        Returns:
            tuple: (x_smooth, y_smooth) coordonnées lissées
        """
        measurement = np.array([x, y])
        
        if not self.initialized:
            # Initialisation avec la première observation
            self.kf.x = np.array([x, y, 0, 0])
            self.initialized = True
        else:
            # Prédiction puis correction
            self.kf.predict()
            self.kf.update(measurement)
        
        return self.kf.x[0], self.kf.x[1]
    
    def predict_future(self, n_steps=10):
        """
        Prédit la trajectoire future
        
        Args:
            n_steps (int): nombre de frames à prédire
            
        Returns:
            list: liste de tuples (x, y) prédits
        """
        if not self.initialized:
            return []
        
        predictions = []
        # Sauvegarde de l'état actuel
        saved_state = self.kf.x.copy()
        saved_P = self.kf.P.copy()
        
        # Prédiction itérative
        for _ in range(n_steps):
            self.kf.predict()
            predictions.append((self.kf.x[0], self.kf.x[1]))
        
        # Restauration de l'état
        self.kf.x = saved_state
        self.kf.P = saved_P
        
        return predictions
    
    def get_velocity(self):
        """
        Retourne la vitesse estimée
        
        Returns:
            tuple: (vx, vy) vitesse en x et y
        """
        if not self.initialized:
            return 0, 0
        return self.kf.x[2], self.kf.x[3]

## 3. Module: Extraction de Caractéristiques (Mouvement)

In [4]:
"""
Module d'extraction des caractéristiques de mouvement
"""
import numpy as np
from collections import deque


class MotionFeatureExtractor:
    """
    Extrait les caractéristiques de mouvement d'un drone:
    - Vitesse instantanée et moyenne
    - Accélération
    - Direction de mouvement
    - Changements brusques
    """
    
    def __init__(self, window_size=10):
        """
        Args:
            window_size (int): taille de la fenêtre pour calculs moyens
        """
        self.window_size = window_size
        self.positions = deque(maxlen=window_size)
        self.velocities = deque(maxlen=window_size)
        self.times = deque(maxlen=window_size)
    
    def add_position(self, x, y, frame):
        """
        Ajoute une nouvelle position
        
        Args:
            x (float): coordonnée x
            y (float): coordonnée y
            frame (int): numéro de frame
        """
        self.positions.append((x, y))
        self.times.append(frame)
        
        # Calcul de la vitesse si on a au moins 2 positions
        if len(self.positions) >= 2:
            dx = self.positions[-1][0] - self.positions[-2][0]
            dy = self.positions[-1][1] - self.positions[-2][1]
            dt = self.times[-1] - self.times[-2]
            
            if dt > 0:
                vx = dx / dt
                vy = dy / dt
                self.velocities.append((vx, vy))
    
    def get_instant_velocity(self):
        """
        Retourne la vitesse instantanée
        
        Returns:
            dict: {'vx', 'vy', 'magnitude', 'direction'}
        """
        if len(self.velocities) == 0:
            return {'vx': 0, 'vy': 0, 'magnitude': 0, 'direction': 0}
        
        vx, vy = self.velocities[-1]
        magnitude = np.sqrt(vx**2 + vy**2)
        direction = np.arctan2(vy, vx) * 180 / np.pi  # en degrés
        
        return {
            'vx': vx,
            'vy': vy,
            'magnitude': magnitude,
            'direction': direction
        }
    
    def get_average_velocity(self):
        """
        Retourne la vitesse moyenne sur la fenêtre
        
        Returns:
            dict: {'vx', 'vy', 'magnitude', 'direction'}
        """
        if len(self.velocities) == 0:
            return {'vx': 0, 'vy': 0, 'magnitude': 0, 'direction': 0}
        
        vx_mean = np.mean([v[0] for v in self.velocities])
        vy_mean = np.mean([v[1] for v in self.velocities])
        magnitude = np.sqrt(vx_mean**2 + vy_mean**2)
        direction = np.arctan2(vy_mean, vx_mean) * 180 / np.pi
        
        return {
            'vx': vx_mean,
            'vy': vy_mean,
            'magnitude': magnitude,
            'direction': direction
        }
    
    def get_acceleration(self):
        """
        Retourne l'accélération instantanée
        
        Returns:
            dict: {'ax', 'ay', 'magnitude'}
        """
        if len(self.velocities) < 2:
            return {'ax': 0, 'ay': 0, 'magnitude': 0}
        
        dvx = self.velocities[-1][0] - self.velocities[-2][0]
        dvy = self.velocities[-1][1] - self.velocities[-2][1]
        dt = self.times[-1] - self.times[-2]
        
        if dt > 0:
            ax = dvx / dt
            ay = dvy / dt
            magnitude = np.sqrt(ax**2 + ay**2)
        else:
            ax = ay = magnitude = 0
        
        return {
            'ax': ax,
            'ay': ay,
            'magnitude': magnitude
        }
    
    def get_direction_change(self):
        """
        Calcule le changement de direction récent
        
        Returns:
            float: changement de direction en degrés
        """
        if len(self.velocities) < 2:
            return 0
        
        # Direction actuelle
        vx1, vy1 = self.velocities[-1]
        angle1 = np.arctan2(vy1, vx1)
        
        # Direction précédente
        vx0, vy0 = self.velocities[-2]
        angle0 = np.arctan2(vy0, vx0)
        
        # Différence d'angle (normalisée entre -180 et 180)
        diff = (angle1 - angle0) * 180 / np.pi
        diff = (diff + 180) % 360 - 180
        
        return abs(diff)
    
    def get_all_features(self):
        """
        Retourne toutes les caractéristiques de mouvement
        
        Returns:
            dict: dictionnaire complet des features
        """
        instant_vel = self.get_instant_velocity()
        avg_vel = self.get_average_velocity()
        accel = self.get_acceleration()
        dir_change = self.get_direction_change()
        
        return {
            'instant_velocity': instant_vel,
            'average_velocity': avg_vel,
            'acceleration': accel,
            'direction_change': dir_change,
            'trajectory_length': len(self.positions)
        }

## 4. Module: Prédiction de Trajectoire (ML)

In [5]:
"""
Module de prédiction de trajectoire avec Machine Learning
"""
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from collections import deque


class TrajectoryPredictor:
    """
    Prédit la trajectoire future d'un drone en utilisant la régression
    """
    
    def __init__(self, history_size=15, degree=2):
        """
        Args:
            history_size (int): nombre de positions historiques utilisées
            degree (int): degré du polynôme pour la régression
        """
        self.history_size = history_size
        self.degree = degree
        
        self.x_positions = deque(maxlen=history_size)
        self.y_positions = deque(maxlen=history_size)
        self.frames = deque(maxlen=history_size)
        
        self.model_x = LinearRegression()
        self.model_y = LinearRegression()
        self.poly_features = PolynomialFeatures(degree=degree)
        
        self.is_trained = False
    
    def add_position(self, x, y, frame):
        """
        Ajoute une nouvelle position à l'historique
        
        Args:
            x (float): coordonnée x
            y (float): coordonnée y
            frame (int): numéro de frame
        """
        self.x_positions.append(x)
        self.y_positions.append(y)
        self.frames.append(frame)
        
        # Entraîner si on a assez de données
        if len(self.frames) >= min(5, self.history_size):
            self._train()
    
    def _train(self):
        """Entraîne les modèles de régression"""
        if len(self.frames) < 3:
            return
        
        # Préparer les données
        frames_array = np.array(self.frames).reshape(-1, 1)
        x_array = np.array(self.x_positions)
        y_array = np.array(self.y_positions)
        
        # Transformation polynomiale
        frames_poly = self.poly_features.fit_transform(frames_array)
        
        # Entraînement
        self.model_x.fit(frames_poly, x_array)
        self.model_y.fit(frames_poly, y_array)
        
        self.is_trained = True
    
    def predict(self, n_steps=10):
        """
        Prédit la trajectoire future
        
        Args:
            n_steps (int): nombre de frames à prédire
            
        Returns:
            list: liste de tuples (x, y) prédits
        """
        if not self.is_trained or len(self.frames) == 0:
            return []
        
        # Frames futures
        last_frame = self.frames[-1]
        future_frames = np.array([last_frame + i + 1 for i in range(n_steps)]).reshape(-1, 1)
        
        # Transformation polynomiale
        future_frames_poly = self.poly_features.transform(future_frames)
        
        # Prédiction
        x_pred = self.model_x.predict(future_frames_poly)
        y_pred = self.model_y.predict(future_frames_poly)
        
        predictions = [(x, y) for x, y in zip(x_pred, y_pred)]
        
        return predictions
    
    def get_predicted_velocity(self):
        """
        Estime la vitesse future basée sur la tendance
        
        Returns:
            tuple: (vx, vy) vitesse prédite
        """
        if not self.is_trained or len(self.frames) < 2:
            return 0, 0
        
        # Prédire 2 points futurs proches
        predictions = self.predict(n_steps=2)
        if len(predictions) < 2:
            return 0, 0
        
        # Calculer la vitesse entre ces 2 points
        dx = predictions[1][0] - predictions[0][0]
        dy = predictions[1][1] - predictions[0][1]
        
        return dx, dy
    
    def get_trajectory_curvature(self):
        """
        Calcule la courbure de la trajectoire
        
        Returns:
            float: mesure de courbure (0 = ligne droite, >0 = courbe)
        """
        if len(self.x_positions) < 3:
            return 0
        
        # Utiliser les 3 derniers points
        positions = list(zip(self.x_positions, self.y_positions))[-3:]
        
        # Calculer les vecteurs
        v1 = np.array([positions[1][0] - positions[0][0], 
                       positions[1][1] - positions[0][1]])
        v2 = np.array([positions[2][0] - positions[1][0], 
                       positions[2][1] - positions[1][1]])
        
        # Éviter division par zéro
        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)
        
        if norm1 < 1e-6 or norm2 < 1e-6:
            return 0
        
        # Produit scalaire normalisé
        cos_angle = np.dot(v1, v2) / (norm1 * norm2)
        cos_angle = np.clip(cos_angle, -1, 1)
        
        # Angle en radians
        angle = np.arccos(cos_angle)
        
        return angle

## 5. Module: Analyse Comportementale

In [6]:
"""
Module de détection de comportements hostiles
Fonctions principales :

analyze(...) → analyse un mouvement et retourne statut, score, raisons.

_determine_status(...) → décide si le drone est normal, suspect ou hostile.

check_proximity_threat(...) → vérifie si le drone s’approche trop près d’une zone sensible.

get_behavior_summary() → fournit un résumé global (moyenne des scores, nombre de comportements hostiles/suspects/normaux).

reset() → réinitialise l’historique.
"""
import numpy as np


class BehaviorAnalyzer:
    """
    Analyse le comportement d'un drone et détecte les patterns hostiles
    """
    
    # Seuils de détection (à ajuster selon le contexte)
    THRESHOLDS = {
        'high_speed': 50,           # vitesse élevée (pixels/frame)
        'high_acceleration': 15,     # accélération élevée
        'sharp_turn': 45,            # changement de direction brusque (degrés)
        'erratic_behavior_count': 3, # nombre de comportements suspects
        'approach_distance': 100,    # distance d'approche critique
        'loitering_frames': 30       # nombre de frames stationnaires
    }
    
    STATUS_NORMAL = "NORMAL"
    STATUS_SUSPECT = "SUSPECT"
    STATUS_HOSTILE = "HOSTILE"
    
    def __init__(self):
        """Initialise l'analyseur de comportement"""
        self.suspicious_events = []
        self.behavior_history = []
    
    def analyze(self, motion_features, trajectory_info=None):
        """
        Analyse le comportement basé sur les features de mouvement
        
        Args:
            motion_features (dict): caractéristiques de mouvement
            trajectory_info (dict): informations additionnelles (optionnel)
            
        Returns:
            dict: {
                'status': NORMAL/SUSPECT/HOSTILE,
                'confidence': float 0-1,
                'reasons': list of str,
                'threat_score': float 0-100
            }
        """
        reasons = []
        threat_score = 0
        
        # Récupération des features
        instant_vel = motion_features.get('instant_velocity', {})
        avg_vel = motion_features.get('average_velocity', {})
        accel = motion_features.get('acceleration', {})
        dir_change = motion_features.get('direction_change', 0)
        
        # 1. Vitesse excessive
        speed = instant_vel.get('magnitude', 0)
        if speed > self.THRESHOLDS['high_speed']:
            reasons.append(f"Vitesse élevée: {speed:.1f} px/frame")
            threat_score += 25
        
        # 2. Accélération brusque
        acceleration = accel.get('magnitude', 0)
        if acceleration > self.THRESHOLDS['high_acceleration']:
            reasons.append(f"Accélération brusque: {acceleration:.1f}")
            threat_score += 20
        
        # 3. Changement de direction soudain
        if dir_change > self.THRESHOLDS['sharp_turn']:
            reasons.append(f"Virage brusque: {dir_change:.1f}°")
            threat_score += 20
            self.suspicious_events.append('sharp_turn')
        
        # 4. Pattern erratique (comportements suspects répétés)
        recent_suspicious = self.suspicious_events[-10:] if self.suspicious_events else []
        if len(recent_suspicious) >= self.THRESHOLDS['erratic_behavior_count']:
            reasons.append("Pattern de mouvement erratique détecté")
            threat_score += 25
        
        # 5. Analyse de la trajectoire (si disponible)
        if trajectory_info:
            curvature = trajectory_info.get('curvature', 0)
            if curvature > 1.0:  # Courbure importante
                reasons.append("Trajectoire en zigzag ou circulaire")
                threat_score += 15
        
        # Détermination du statut
        status = self._determine_status(threat_score, len(reasons))
        confidence = min(threat_score / 100, 1.0)
        
        # Historique
        self.behavior_history.append({
            'threat_score': threat_score,
            'status': status,
            'reasons': reasons
        })
        
        return {
            'status': status,
            'confidence': confidence,
            'reasons': reasons,
            'threat_score': threat_score
        }
    
    def _determine_status(self, threat_score, num_reasons):
        """
        Détermine le statut du drone
        
        Args:
            threat_score (float): score de menace
            num_reasons (int): nombre de raisons suspectes
            
        Returns:
            str: NORMAL, SUSPECT ou HOSTILE
        """
        if threat_score >= 60 or num_reasons >= 3:
            return self.STATUS_HOSTILE
        elif threat_score >= 30 or num_reasons >= 2:
            return self.STATUS_SUSPECT
        else:
            return self.STATUS_NORMAL
    
    def check_proximity_threat(self, drone_pos, sensitive_areas=None):
        """
        Vérifie si le drone s'approche de zones sensibles
        
        Args:
            drone_pos (tuple): position (x, y) du drone
            sensitive_areas (list): liste de zones [(x, y, radius), ...]
            
        Returns:
            dict: informations sur la menace de proximité
        """
        if not sensitive_areas:
            return {'is_threat': False, 'distance': None}
        
        min_distance = float('inf')
        closest_area = None
        
        for i, (area_x, area_y, radius) in enumerate(sensitive_areas):
            distance = np.sqrt((drone_pos[0] - area_x)**2 + 
                             (drone_pos[1] - area_y)**2)
            
            if distance < min_distance:
                min_distance = distance
                closest_area = i
        
        is_threat = min_distance < self.THRESHOLDS['approach_distance']
        
        return {
            'is_threat': is_threat,
            'distance': min_distance,
            'closest_area_index': closest_area
        }
    
    def get_behavior_summary(self):
        """
        Retourne un résumé du comportement global
        
        Returns:
            dict: statistiques sur le comportement
        """
        if not self.behavior_history:
            return {
                'average_threat_score': 0,
                'max_threat_score': 0,
                'hostile_count': 0,
                'suspect_count': 0,
                'normal_count': 0
            }
        
        threat_scores = [b['threat_score'] for b in self.behavior_history]
        statuses = [b['status'] for b in self.behavior_history]
        
        return {
            'average_threat_score': np.mean(threat_scores),
            'max_threat_score': np.max(threat_scores),
            'hostile_count': statuses.count(self.STATUS_HOSTILE),
            'suspect_count': statuses.count(self.STATUS_SUSPECT),
            'normal_count': statuses.count(self.STATUS_NORMAL)
        }
    
    def reset(self):
        """Réinitialise l'historique"""
        self.suspicious_events = []
        self.behavior_history = []

## 6. Module: Visualisation

In [7]:
"""
Module de visualisation des résultats
"""
import cv2
import numpy as np
import winsound  # Pour l'alarme sonore sur Windows
import threading


class DroneVisualizer:
    """
    Gère la visualisation des trajectoires et statuts des drones
    """
    
    # Couleurs selon le statut (BGR pour OpenCV)
    COLORS = {
        'NORMAL': (0, 255, 0),      # Vert
        'SUSPECT': (0, 165, 255),   # Orange
        'HOSTILE': (0, 0, 255)      # Rouge
    }
    
    def __init__(self, trail_length=30, restricted_zone=None):
        """
        Args:
            trail_length (int): longueur de la traînée de trajectoire
            restricted_zone (list): liste de points définissant la zone restreinte [(x1,y1), (x2,y2), ...]
        """
        self.trail_length = trail_length
        self.restricted_zone = restricted_zone or []
        self.drones_in_zone = set()  # Track des drones actuellement dans la zone
        self.alarm_playing = False  # Indique si l'alarme est en cours de lecture
        self.alarm_thread = None  # Thread pour l'alarme continue
    
    def draw_drone_info(self, frame, drone_data):
        """
        Dessine les informations d'un drone sur la frame
        
        Args:
            frame (np.array): image de la frame
            drone_data (dict): données du drone contenant:
                - track_id
                - x, y (position actuelle)
                - status
                - trajectory (liste de positions historiques)
                - predicted_trajectory (liste de positions futures)
                - confidence
                - threat_score
                - velocity
                
        Returns:
            np.array: frame avec annotations
        """
        track_id = drone_data.get('track_id')
        x, y = int(drone_data.get('x')), int(drone_data.get('y'))
        status = drone_data.get('status', 'NORMAL')
        trajectory = drone_data.get('trajectory', [])
        predicted = drone_data.get('predicted_trajectory', [])
        confidence = drone_data.get('confidence', 0)
        threat_score = drone_data.get('threat_score', 0)
        velocity = drone_data.get('velocity', {})
        behavior = drone_data.get('behavior', {})
        reasons = behavior.get('reasons', [])
        
        color = self.COLORS.get(status, (255, 255, 255))
        
        # 1. Dessiner la trajectoire historique (seulement jusqu'à la frame actuelle)
        # La trajectoire s'affiche progressivement au fur et à mesure
        if len(trajectory) > 1:
            self._draw_trajectory(frame, trajectory, color, alpha=0.6)
        
        # 2. Dessiner la trajectoire prédite (pointillés) - seulement si on a assez de données
        if len(trajectory) >= 3:  # Nécessite au moins 3 points pour prédire
            self._draw_predicted_trajectory(frame, predicted, color)
        
        # 3. Dessiner un CARRE COLORÉ selon l'état (autour du drone)
        # Le carré n'apparaît que quand le drone est détecté à cette frame
        # Taille du carré pour bien entourer le drone (ajustable selon la taille du drone)
        # Utiliser une taille plus grande pour mieux entourer l'objet
        square_size = 80  # Taille augmentée pour mieux entourer le drone
        top_left = (x - square_size // 2, y - square_size // 2)
        bottom_right = (x + square_size // 2, y + square_size // 2)
        
        # Dessiner le carré avec une bordure épaisse et colorée selon l'état
        cv2.rectangle(frame, top_left, bottom_right, color, 4)
        
        # Remplir le carré avec transparence (pour voir le drone à travers)
        overlay = frame.copy()
        cv2.rectangle(overlay, top_left, bottom_right, color, -1)
        cv2.addWeighted(overlay, 0.15, frame, 0.85, 0, frame)
        
        # Dessiner une bordure extérieure pour meilleure visibilité
        cv2.rectangle(frame, top_left, bottom_right, color, 4)
        
        # 4. Dessiner la position actuelle (cercle au centre)
        cv2.circle(frame, (x, y), 6, color, -1)
        cv2.circle(frame, (x, y), 8, color, 2)
        
        # 5. Dessiner un panneau d'information EN HAUT du carré avec position, état et vitesse
        # Centrer le panneau au-dessus du carré
        info_panel_width = 200
        info_panel_height = 85  # Augmenté pour inclure la vitesse
        info_panel_x = x - info_panel_width // 2
        info_panel_y = y - square_size // 2 - info_panel_height - 10
        
        # S'assurer que le panneau ne sort pas de l'image
        height, width = frame.shape[:2]
        if info_panel_x < 0:
            info_panel_x = 5
        if info_panel_x + info_panel_width > width:
            info_panel_x = width - info_panel_width - 5
        if info_panel_y < 0:
            info_panel_y = y + square_size // 2 + 10  # Mettre en bas si pas de place en haut
        
        # Fond du panneau (noir opaque pour meilleure lisibilité)
        cv2.rectangle(frame,
                     (info_panel_x, info_panel_y),
                     (info_panel_x + info_panel_width, info_panel_y + info_panel_height),
                     (0, 0, 0), -1)
        
        # Bordure du panneau avec la couleur de l'état
        cv2.rectangle(frame,
                     (info_panel_x, info_panel_y),
                     (info_panel_x + info_panel_width, info_panel_y + info_panel_height),
                     color, 2)
        
        # Texte dans le panneau : Position, État et Vitesse
        text_y = info_panel_y + 18
        # ID du drone
        self._draw_text(frame, f"Drone ID: {track_id}", 
                      (info_panel_x + 8, text_y), 
                      (255, 255, 255), font_scale=0.5, thickness=1)
        
        text_y += 18
        # Position (x, y) du CSV
        self._draw_text(frame, f"Position: X={int(x)}, Y={int(y)}", 
                      (info_panel_x + 8, text_y), 
                      (255, 255, 255), font_scale=0.5, thickness=1)
        
        text_y += 18
        # Vitesse (affichée à l'extérieur de la zone)
        velocity_magnitude = velocity.get('magnitude', 0)
        self._draw_text(frame, f"Vitesse: {velocity_magnitude:.1f} px/f", 
                      (info_panel_x + 8, text_y), 
                      (255, 255, 255), font_scale=0.5, thickness=1)
        
        text_y += 18
        # État avec couleur
        self._draw_text(frame, f"Etat: {status}", 
                      (info_panel_x + 8, text_y), 
                      color, font_scale=0.6, thickness=2)
        
        # 6. Ligne de connexion entre le panneau et le carré
        line_start_y = y - square_size // 2
        line_end_y = info_panel_y + info_panel_height
        cv2.line(frame, 
                (x, line_start_y),
                (x, line_end_y),
                color, 2)
        
        # 7. Dessiner un indicateur de direction
        if velocity.get('magnitude', 0) > 1:
            self._draw_velocity_arrow(frame, x, y, velocity, color)
        
        return frame
    
    def _point_in_polygon(self, point, polygon):
        """
        Vérifie si un point est dans un polygone en utilisant le ray casting algorithm
        
        Args:
            point (tuple): (x, y) point à tester
            polygon (list): liste de points [(x1,y1), (x2,y2), ...] définissant le polygone
            
        Returns:
            bool: True si le point est dans le polygone
        """
        if len(polygon) < 3:
            return False
        
        x, y = point
        n = len(polygon)
        inside = False
        
        p1x, p1y = polygon[0]
        for i in range(1, n + 1):
            p2x, p2y = polygon[i % n]
            if y > min(p1y, p2y):
                if y <= max(p1y, p2y):
                    if x <= max(p1x, p2x):
                        if p1y != p2y:
                            xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                        if p1x == p2x or x <= xinters:
                            inside = not inside
            p1x, p1y = p2x, p2y
        
        return inside
    
    def _play_alarm_continuous(self):
        """Joue une alarme sonore en continu"""
        try:
            # Fréquence de l'alarme
            frequency = 1000  # Hz
            duration = 300  # ms par beep
            pause = 200  # ms de pause entre les beeps
            
            # Jouer l'alarme en boucle tant que alarm_playing est True
            while self.alarm_playing:
                winsound.Beep(frequency, duration)
                import time
                time.sleep(pause / 1000.0)  # Convertir ms en secondes
        except:
            pass  # Ignorer les erreurs si l'alarme ne peut pas être jouée
    
    def check_restricted_zone(self, frame, drone_positions):
        """
        Vérifie si un drone est dans la zone restreinte et applique les effets
        
        Args:
            frame (np.array): frame de la vidéo
            drone_positions (list): liste de positions de drones [(x, y, track_id), ...]
            
        Returns:
            tuple: (frame modifié, zone_violated)
        """
        zone_violated = False
        
        if not self.restricted_zone or len(self.restricted_zone) < 3:
            return frame, zone_violated
        
        # Vérifier chaque drone
        current_drones_in_zone = set()
        for x, y, track_id in drone_positions:
            if self._point_in_polygon((x, y), self.restricted_zone):
                zone_violated = True
                current_drones_in_zone.add(track_id)
                
                # Démarrer l'alarme continue si elle n'est pas déjà en cours
                if not self.alarm_playing:
                    self.alarm_playing = True
                    # Démarrer l'alarme dans un thread séparé
                    if self.alarm_thread is None or not self.alarm_thread.is_alive():
                        self.alarm_thread = threading.Thread(target=self._play_alarm_continuous, daemon=True)
                        self.alarm_thread.start()
                
                # Effet d'écran rouge
                red_overlay = frame.copy()
                red_overlay[:, :] = (0, 0, 255)  # Rouge en BGR
                frame = cv2.addWeighted(frame, 0.3, red_overlay, 0.7, 0)
                
                # Message d'alerte
                height, width = frame.shape[:2]
                alert_text = "ALERTE: ZONE RESTREINTE VIOLEE PAR DRONE!"
                alert_text2 = f"Drone ID {track_id} est dans la zone restreinte"
                
                # Fond pour le message
                text_size = cv2.getTextSize(alert_text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)[0]
                text_x = (width - text_size[0]) // 2
                text_y = height // 2 - 30
                
                cv2.rectangle(frame, 
                             (text_x - 20, text_y - 50),
                             (text_x + text_size[0] + 20, text_y + 80),
                             (0, 0, 0), -1)
                
                cv2.rectangle(frame, 
                             (text_x - 20, text_y - 50),
                             (text_x + text_size[0] + 20, text_y + 80),
                             (0, 0, 255), 5)
                
                # Texte d'alerte
                cv2.putText(frame, alert_text, (text_x, text_y),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3, cv2.LINE_AA)
                
                cv2.putText(frame, alert_text2, (text_x, text_y + 40),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
                
                break  # Un seul drone dans la zone suffit pour déclencher l'alerte
        
        # Arrêter l'alarme si aucun drone n'est dans la zone
        if not zone_violated and self.alarm_playing:
            self.alarm_playing = False
        
        # Mettre à jour la liste des drones dans la zone
        self.drones_in_zone = current_drones_in_zone
        
        return frame, zone_violated
    
    def _get_brief_analysis(self, status, threat_score, reasons):
        """
        Génère une analyse brève basée sur le statut et les raisons
        
        Args:
            status (str): statut du drone
            threat_score (float): score de menace
            reasons (list): liste des raisons
            
        Returns:
            list: lignes de texte pour l'analyse
        """
        analysis = []
        
        if status == 'NORMAL':
            analysis.append("Analyse: Comportement")
            analysis.append("normal detecte")
        elif status == 'SUSPECT':
            analysis.append("Analyse: Comportement")
            analysis.append("suspect detecte")
            if reasons:
                # Prendre la première raison
                first_reason = reasons[0][:30]  # Limiter la longueur
                analysis.append(first_reason)
        else:  # HOSTILE
            analysis.append("Analyse: Comportement")
            analysis.append("HOSTILE detecte!")
            if reasons:
                # Prendre les 2 premières raisons
                for reason in reasons[:2]:
                    short_reason = reason[:30]
                    analysis.append(short_reason)
        
        return analysis
    
    def _draw_trajectory(self, frame, trajectory, color, alpha=0.6):
        """Dessine la trajectoire historique"""
        if len(trajectory) < 2:
            return
        
        # Limiter à la longueur de traînée
        recent_trajectory = trajectory[-self.trail_length:]
        
        # Dessiner les lignes
        for i in range(len(recent_trajectory) - 1):
            pt1 = (int(recent_trajectory[i][0]), int(recent_trajectory[i][1]))
            pt2 = (int(recent_trajectory[i+1][0]), int(recent_trajectory[i+1][1]))
            
            # Épaisseur décroissante
            thickness = max(1, int(3 * (i + 1) / len(recent_trajectory)))
            
            # Transparence croissante
            trail_color = tuple(int(c * (0.3 + 0.7 * (i + 1) / len(recent_trajectory))) 
                              for c in color)
            
            cv2.line(frame, pt1, pt2, trail_color, thickness)
    
    def _draw_predicted_trajectory(self, frame, predicted, color):
        """Dessine la trajectoire prédite en pointillés"""
        if len(predicted) < 2:
            return
        
        for i in range(len(predicted) - 1):
            pt1 = (int(predicted[i][0]), int(predicted[i][1]))
            pt2 = (int(predicted[i+1][0]), int(predicted[i+1][1]))
            
            # Ligne en pointillés
            self._draw_dashed_line(frame, pt1, pt2, color, thickness=2)
            
            # Petit cercle à chaque point prédit
            cv2.circle(frame, pt2, 3, color, -1)
    
    def _draw_dashed_line(self, frame, pt1, pt2, color, thickness=1, dash_length=10):
        """Dessine une ligne en pointillés"""
        dist = np.sqrt((pt2[0] - pt1[0])**2 + (pt2[1] - pt1[1])**2)
        if dist < 1:
            return
        
        dashes = int(dist / dash_length)
        for i in range(0, dashes, 2):
            start_ratio = i / dashes
            end_ratio = min((i + 1) / dashes, 1.0)
            
            start_pt = (
                int(pt1[0] + (pt2[0] - pt1[0]) * start_ratio),
                int(pt1[1] + (pt2[1] - pt1[1]) * start_ratio)
            )
            end_pt = (
                int(pt1[0] + (pt2[0] - pt1[0]) * end_ratio),
                int(pt1[1] + (pt2[1] - pt1[1]) * end_ratio)
            )
            
            cv2.line(frame, start_pt, end_pt, color, thickness)
    
    def _draw_velocity_arrow(self, frame, x, y, velocity, color):
        """Dessine une flèche indiquant la direction"""
        vx = velocity.get('vx', 0)
        vy = velocity.get('vy', 0)
        
        # Normaliser et agrandir
        scale = 30
        end_x = int(x + vx * scale / max(abs(vx) + abs(vy), 1))
        end_y = int(y + vy * scale / max(abs(vx) + abs(vy), 1))
        
        cv2.arrowedLine(frame, (x, y), (end_x, end_y), color, 2, 
                       tipLength=0.3)
    
    def _draw_text(self, frame, text, position, color, font_scale=0.5, 
                   thickness=1):
        """Dessine du texte avec fond pour meilleure lisibilité"""
        font = cv2.FONT_HERSHEY_SIMPLEX
        
        # Fond noir semi-transparent
        (text_width, text_height), baseline = cv2.getTextSize(
            text, font, font_scale, thickness
        )
        
        x, y = position
        cv2.rectangle(frame, 
                     (x - 2, y - text_height - 2),
                     (x + text_width + 2, y + baseline + 2),
                     (0, 0, 0), -1)
        
        # Texte
        cv2.putText(frame, text, (x, y), font, font_scale, color, 
                   thickness, cv2.LINE_AA)
    
    def draw_restricted_zone(self, frame):
        """
        Dessine la zone restreinte sur la frame
        
        Args:
            frame (np.array): frame de la vidéo
            
        Returns:
            np.array: frame avec la zone restreinte dessinée
        """
        if not self.restricted_zone or len(self.restricted_zone) < 3:
            return frame
        
        # Convertir en numpy array pour cv2.fillPoly
        pts = np.array(self.restricted_zone, np.int32)
        pts = pts.reshape((-1, 1, 2))
        
        # Dessiner la zone en rouge semi-transparent
        overlay = frame.copy()
        cv2.fillPoly(overlay, [pts], (0, 0, 255))  # Rouge en BGR
        frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)
        
        # Dessiner le contour en rouge épais
        cv2.polylines(frame, [pts], True, (0, 0, 255), 3)
        
        return frame
    
    def draw_legend(self, frame):
        """Dessine une légende EN HAUT de la frame"""
        height, width = frame.shape[:2]
        
        # Légende en haut au centre
        legend_width = 400
        legend_x = (width - legend_width) // 2
        legend_y = 20
        
        # Fond de la légende (semi-transparent)
        overlay = frame.copy()
        cv2.rectangle(overlay, 
                     (legend_x - 10, legend_y - 10),
                     (legend_x + legend_width, legend_y + 80),
                     (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
        
        # Bordure
        cv2.rectangle(frame, 
                     (legend_x - 10, legend_y - 10),
                     (legend_x + legend_width, legend_y + 80),
                     (255, 255, 255), 2)
        
        # Titre
        self._draw_text(frame, "LEGENDE - ETAT DU DRONE", (legend_x, legend_y), 
                       (255, 255, 255), font_scale=0.7, thickness=2)
        
        # Statuts avec carrés colorés
        statuses = [
            ('NORMAL', 'NORMAL - Comportement normal'),
            ('SUSPECT', 'SUSPECT - Comportement suspect'),
            ('HOSTILE', 'HOSTILE - Comportement hostile')
        ]
        
        y_offset = legend_y + 30
        square_size = 15
        for status_key, status_label in statuses:
            color = self.COLORS[status_key]
            # Dessiner un carré
            top_left = (legend_x + 10, y_offset - square_size // 2)
            bottom_right = (legend_x + 10 + square_size, y_offset + square_size // 2)
            cv2.rectangle(frame, top_left, bottom_right, color, -1)
            cv2.rectangle(frame, top_left, bottom_right, (255, 255, 255), 1)
            
            self._draw_text(frame, status_label, (legend_x + 35, y_offset + 5), 
                           (255, 255, 255), font_scale=0.5)
            y_offset += 20
    
    def create_summary_overlay(self, frame, drones_data):
        """
        Crée un overlay avec des statistiques globales
        
        Args:
            frame (np.array): frame
            drones_data (list): liste de données de tous les drones
        """
        height, width = frame.shape[:2]
        
        # Comptage par statut
        status_counts = {'NORMAL': 0, 'SUSPECT': 0, 'HOSTILE': 0}
        for drone in drones_data:
            status = drone.get('status', 'NORMAL')
            status_counts[status] = status_counts.get(status, 0) + 1
        
        # Overlay en haut à gauche
        overlay_x = 20
        overlay_y = 30
        
        # Fond
        cv2.rectangle(frame, 
                     (overlay_x - 10, overlay_y - 20),
                     (overlay_x + 250, overlay_y + 90),
                     (0, 0, 0), -1)
        
        # Titre
        self._draw_text(frame, f"DRONES DETECTES: {len(drones_data)}", 
                       (overlay_x, overlay_y), (255, 255, 255), 
                       font_scale=0.6, thickness=2)
        
        # Compteurs
        y_offset = overlay_y + 25
        for status, count in status_counts.items():
            color = self.COLORS[status]
            text = f"{status}: {count}"
            cv2.circle(frame, (overlay_x + 10, y_offset), 6, color, -1)
            self._draw_text(frame, text, (overlay_x + 25, y_offset + 5), 
                           (255, 255, 255), font_scale=0.5)
            y_offset += 20
        
        return frame
    
    @staticmethod
    def plot_trajectory_matplotlib(results_data, output_path="trajectory_plot.png"):
        """
        Génère une visualisation matplotlib de la trajectoire
        
        Args:
            results_data (dict): dictionnaire avec track_id comme clé et liste de résultats comme valeur
            output_path (str): chemin de sortie pour l'image
        """
        import matplotlib.pyplot as plt
        import matplotlib.patches as mpatches
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Couleurs pour matplotlib (RGB au lieu de BGR)
        colors_map = {
            'NORMAL': (0, 1, 0),      # Vert
            'SUSPECT': (1, 0.65, 0),   # Orange
            'HOSTILE': (1, 0, 0)       # Rouge
        }
        
        # Parcourir tous les drones
        for track_id, results in results_data.items():
            if not results:
                continue
            
            # Extraire les positions
            x_coords = [r['x_smooth'] for r in results]
            y_coords = [r['y_smooth'] for r in results]
            statuses = [r['status'] for r in results]
            frames = [r['frame'] for r in results]
            
            # Tracer la trajectoire avec des couleurs selon le statut
            for i in range(len(x_coords) - 1):
                status = statuses[i]
                color = colors_map.get(status, (0.5, 0.5, 0.5))
                
                # Ligne entre les points
                ax.plot([x_coords[i], x_coords[i+1]], 
                       [y_coords[i], y_coords[i+1]], 
                       color=color, linewidth=2, alpha=0.7)
            
            # Marquer les points avec des couleurs selon le statut
            for i, (x, y, status) in enumerate(zip(x_coords, y_coords, statuses)):
                color = colors_map.get(status, (0.5, 0.5, 0.5))
                marker_size = 50 if i == 0 else (80 if i == len(x_coords) - 1 else 30)
                marker = 'o' if i == 0 else ('s' if i == len(x_coords) - 1 else 'o')
                
                ax.scatter(x, y, c=[color], s=marker_size, marker=marker, 
                          edgecolors='black', linewidths=1.5, alpha=0.8)
            
            # Annoter le point de départ et d'arrivée
            if len(x_coords) > 0:
                ax.annotate(f'Début ID:{track_id}', 
                           (x_coords[0], y_coords[0]),
                           xytext=(10, 10), textcoords='offset points',
                           fontsize=9, bbox=dict(boxstyle='round,pad=0.3', 
                                                facecolor='yellow', alpha=0.7))
                ax.annotate(f'Fin ID:{track_id}', 
                           (x_coords[-1], y_coords[-1]),
                           xytext=(10, -20), textcoords='offset points',
                           fontsize=9, bbox=dict(boxstyle='round,pad=0.3', 
                                                facecolor='lightblue', alpha=0.7))
        
        # Configuration du graphique
        ax.set_xlabel('Position X (pixels)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Position Y (pixels)', fontsize=12, fontweight='bold')
        ax.set_title('Visualisation de la Trajectoire du Drone', 
                    fontsize=14, fontweight='bold', pad=20)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_aspect('equal', adjustable='box')
        
        # Légende
        legend_elements = [
            mpatches.Patch(facecolor=colors_map['NORMAL'], 
                          label='NORMAL - Comportement normal'),
            mpatches.Patch(facecolor=colors_map['SUSPECT'], 
                          label='SUSPECT - Comportement suspect'),
            mpatches.Patch(facecolor=colors_map['HOSTILE'], 
                          label='HOSTILE - Comportement hostile'),
            plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black',
                      markersize=8, label='Point de départ'),
            plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='black',
                      markersize=8, label='Point d\'arrivée')
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"Visualisation matplotlib sauvegardée: {output_path}")

## 7. Orchestration Principale

In [8]:
"""
Script principal d'orchestration du système d'analyse de trajectoire
"""
import cv2
import json
import csv
import numpy as np
from pathlib import Path
from collections import defaultdict
import wave
import struct
import subprocess
import os

# Imports des modules du projet
# from trajectory_analysis.kalman import DroneKalmanFilter
# from trajectory_analysis.motion_features import MotionFeatureExtractor
# from trajectory_analysis.trajectoire_ML import TrajectoryPredictor
# from trajectory_analysis.behavior_analysis import BehaviorAnalyzer
# from visualization.results import DroneVisualizer


class DroneTrajectoryAnalyzer:
    """
    Système complet d'analyse de trajectoire et détection comportementale
    """
    
    def __init__(self, config=None):
        """
        Args:
            config (dict): configuration du système
        """
        self.config = config or {}
        
        # Trackers par drone (un par track_id)
        self.kalman_filters = {}
        self.motion_extractors = {}
        self.trajectory_predictors = {}
        self.behavior_analyzers = {}
        
        # Visualiseur avec zone restreinte
        restricted_zone = self.config.get('restricted_zone', None)
        self.visualizer = DroneVisualizer(
            trail_length=self.config.get('trail_length', 30),
            restricted_zone=restricted_zone
        )
        
        # Stockage des résultats
        self.results = defaultdict(list)
        
        # Paramètres
        self.prediction_steps = self.config.get('prediction_steps', 10)
    
    def process_tracking_data(self, track_id, frame_num, x, y):
        """
        Traite les données de tracking pour un drone
        
        Args:
            track_id (int): identifiant du drone
            frame_num (int): numéro de frame
            x (float): coordonnée x
            y (float): coordonnée y
            
        Returns:
            dict: résultats de l'analyse pour cette frame
        """
        # Initialisation des modules pour ce drone si nécessaire
        if track_id not in self.kalman_filters:
            self._initialize_drone_trackers(track_id)
        
        # 1. Lissage avec Kalman
        x_smooth, y_smooth = self.kalman_filters[track_id].update(x, y)
        
        # 2. Extraction des features de mouvement
        self.motion_extractors[track_id].add_position(x_smooth, y_smooth, frame_num)
        motion_features = self.motion_extractors[track_id].get_all_features()
        
        # 3. Prédiction de trajectoire avec ML
        self.trajectory_predictors[track_id].add_position(x_smooth, y_smooth, frame_num)
        predicted_trajectory = self.trajectory_predictors[track_id].predict(
            n_steps=self.prediction_steps
        )
        trajectory_curvature = self.trajectory_predictors[track_id].get_trajectory_curvature()
        
        # 4. Analyse comportementale
        trajectory_info = {'curvature': trajectory_curvature}
        behavior_result = self.behavior_analyzers[track_id].analyze(
            motion_features, trajectory_info
        )
        
        # 5. Prédiction Kalman (alternative/complémentaire)
        kalman_predictions = self.kalman_filters[track_id].predict_future(
            n_steps=self.prediction_steps
        )
        
        # Compilation des résultats
        result = {
            'track_id': track_id,
            'frame': frame_num,
            'x': x,
            'y': y,
            'x_smooth': x_smooth,
            'y_smooth': y_smooth,
            'motion_features': motion_features,
            'predicted_trajectory': predicted_trajectory,
            'kalman_predictions': kalman_predictions,
            'behavior': behavior_result,
            'status': behavior_result['status'],
            'confidence': behavior_result['confidence'],
            'threat_score': behavior_result['threat_score'],
            'trajectory_curvature': trajectory_curvature
        }
        
        # Stockage
        self.results[track_id].append(result)
        
        return result
    
    def _initialize_drone_trackers(self, track_id):
        """Initialise les trackers pour un nouveau drone"""
        self.kalman_filters[track_id] = DroneKalmanFilter()
        self.motion_extractors[track_id] = MotionFeatureExtractor(window_size=10)
        self.trajectory_predictors[track_id] = TrajectoryPredictor(
            history_size=15, degree=2
        )
        self.behavior_analyzers[track_id] = BehaviorAnalyzer()
    
    def process_video(self, video_path, tracking_data_path, output_video_path):
        """
        Traite une vidéo complète avec les données de tracking
        
        Args:
            video_path (str): chemin vers la vidéo
            tracking_data_path (str): chemin vers les données de tracking (CSV)
            output_video_path (str): chemin de sortie de la vidéo annotée
        """
        # Chargement des données de tracking
        tracking_data = self._load_tracking_data(tracking_data_path)
        
        # Ouverture de la vidéo
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Impossible d'ouvrir la vidéo: {video_path}")
        
        # Propriétés de la vidéo
        fps = int(cap.get(cv2.CAP_PROP_FPS))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Writer pour la vidéo de sortie
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
        
        # Créer un mapping des frames de tracking aux frames vidéo
        # Pour que chaque position soit affichée suffisamment longtemps
        tracking_frames = sorted(tracking_data.keys())
        num_tracking_frames = len(tracking_frames)
        
        if num_tracking_frames == 0:
            raise ValueError("Aucune donnée de tracking trouvée dans le CSV")
        
        # Calculer combien de frames vidéo correspondent à chaque position
        # Pour que l'affichage soit fluide et lisible
        frames_per_position = max(1, total_frames // num_tracking_frames)
        
        print(f"Traitement de la vidéo: {total_frames} frames vidéo, {num_tracking_frames} positions de tracking")
        print(f"Chaque position sera affichée pendant environ {frames_per_position} frames")
        
        # Créer un mapping: frame vidéo -> frame de tracking
        video_to_tracking = {}
        for i, tracking_frame in enumerate(tracking_frames):
            start_video_frame = i * frames_per_position
            end_video_frame = min((i + 1) * frames_per_position, total_frames)
            
            # Pour toutes les frames vidéo dans cette plage, utiliser cette position de tracking
            for vf in range(start_video_frame, end_video_frame):
                video_to_tracking[vf] = tracking_frame
        
        # S'assurer que les dernières frames utilisent la dernière position
        for vf in range(len(video_to_tracking), total_frames):
            video_to_tracking[vf] = tracking_frames[-1]
        
        frame_num = 0
        last_frame = None
        alarm_frames = []  # Enregistrer les frames où l'alarme doit sonner
        
        # Traiter toutes les frames de la vidéo
        while frame_num < total_frames:
            # Lire une frame de la vidéo
            ret, frame = cap.read()
            if not ret:
                break
            
            last_frame = frame.copy()
            
            # Traitement des drones dans cette frame
            current_drones = []
            drone_positions = []  # Pour la détection de zone
            
            # Obtenir la frame de tracking correspondante à cette frame vidéo
            tracking_frame_idx = video_to_tracking.get(frame_num)
            
            if tracking_frame_idx is not None and tracking_frame_idx in tracking_data:
                for detection in tracking_data[tracking_frame_idx]:
                    track_id = detection['track_id']
                    x, y = detection['x'], detection['y']
                    
                    # Stocker la position pour la détection de zone
                    drone_positions.append((int(x), int(y), track_id))
                    
                    # Analyse (utiliser le numéro de frame de tracking pour l'analyse)
                    result = self.process_tracking_data(track_id, tracking_frame_idx, x, y)
                    
                    # Préparation pour visualisation
                    # Utiliser frame_num (frame vidéo) pour l'affichage progressif
                    drone_viz_data = self._prepare_visualization_data(result, frame_num)
                    current_drones.append(drone_viz_data)
                    
                    # Dessin sur la frame (le carré et les infos apparaissent progressivement)
                    frame = self.visualizer.draw_drone_info(frame, drone_viz_data)
            
            # Dessiner la zone restreinte sur la frame
            frame = self.visualizer.draw_restricted_zone(frame)
            
            # Vérifier si un drone est dans la zone restreinte
            frame, zone_violated = self.visualizer.check_restricted_zone(frame, drone_positions)
            
            # Enregistrer si l'alarme doit sonner à cette frame
            if zone_violated:
                alarm_frames.append(frame_num)
            
            # Overlay avec statistiques
            frame = self.visualizer.create_summary_overlay(frame, current_drones)
            
            # Légende
            self.visualizer.draw_legend(frame)
            
            # Écriture
            out.write(frame)
            
            frame_num += 1
            if frame_num % 30 == 0:
                print(f"Progression: {frame_num}/{total_frames} frames")
        
        cap.release()
        out.release()
        
        print(f"Vidéo annotée sauvegardée (sans audio): {output_video_path}")
        print(f"Total de frames traitées: {frame_num}")
        
        # Générer et ajouter l'audio d'alarme (même si vide, pour avoir une piste audio)
        print(f"Génération de l'audio d'alarme pour {len(alarm_frames)} frames...")
        self._add_alarm_audio_to_video(output_video_path, fps, total_frames, alarm_frames, video_path)
    
    def _generate_alarm_audio(self, output_path, fps, total_frames, alarm_frames):
        """
        Génère un fichier audio WAV avec l'alarme aux moments où la zone est violée
        
        Args:
            output_path (str): chemin de sortie du fichier audio
            fps (int): frames par seconde de la vidéo
            total_frames (int): nombre total de frames
            alarm_frames (list): liste des numéros de frames où l'alarme doit sonner
        """
        sample_rate = 44100  # Fréquence d'échantillonnage
        frequency = 1000  # Fréquence de l'alarme en Hz
        beep_duration = 0.3  # Durée d'un beep en secondes
        total_duration = total_frames / fps  # Durée totale de la vidéo
        
        # Créer un tableau de silence
        num_samples = int(sample_rate * total_duration)
        audio_data = np.zeros(num_samples, dtype=np.float32)
        
        # Ajouter l'alarme aux frames où la zone est violée
        for frame_num in alarm_frames:
            time_in_video = frame_num / fps
            
            # Générer un beep pour chaque frame d'alarme
            start_sample = int(time_in_video * sample_rate)
            end_sample = min(int((time_in_video + beep_duration) * sample_rate), num_samples)
            
            # Générer le beep (onde sinusoïdale)
            duration_samples = end_sample - start_sample
            if duration_samples > 0:
                t = np.linspace(0, beep_duration, duration_samples)
                beep = np.sin(2 * np.pi * frequency * t)
                
                # Appliquer une enveloppe pour éviter les clics
                envelope = np.ones_like(beep)
                fade_samples = int(0.01 * sample_rate)  # 10ms de fade
                if fade_samples < len(envelope):
                    envelope[:fade_samples] = np.linspace(0, 1, fade_samples)
                    envelope[-fade_samples:] = np.linspace(1, 0, fade_samples)
                beep *= envelope
                
                # Ajouter le beep à l'audio (mixer avec l'existant)
                if start_sample < num_samples:
                    audio_data[start_sample:end_sample] += beep * 0.5
        
        # Normaliser l'audio
        max_val = np.max(np.abs(audio_data))
        if max_val > 0:
            audio_data = audio_data / max_val * 0.8  # Volume à 80%
        
        # Convertir en int16 pour WAV
        audio_int16 = (audio_data * 32767).astype(np.int16)
        
        # Écrire le fichier WAV
        with wave.open(output_path, 'w') as wav_file:
            wav_file.setnchannels(1)  # Mono
            wav_file.setsampwidth(2)  # 16 bits
            wav_file.setframerate(sample_rate)
            wav_file.writeframes(audio_int16.tobytes())
        
        print(f"Audio d'alarme généré: {output_path}")
    
    def _add_alarm_audio_to_video(self, video_path, fps, total_frames, alarm_frames, original_video_path=None):
        """
        Ajoute l'audio d'alarme à la vidéo et conserve l'audio original si présent
        
        Args:
            video_path (str): chemin de la vidéo de sortie
            fps (int): frames par seconde
            total_frames (int): nombre total de frames
            alarm_frames (list): liste des frames où l'alarme doit sonner
            original_video_path (str): chemin de la vidéo originale (pour extraire l'audio)
        """
        try:
            # Générer le fichier audio temporaire de l'alarme
            temp_audio = video_path.replace('.mp4', '_alarm_temp.wav')
            self._generate_alarm_audio(temp_audio, fps, total_frames, alarm_frames)
            
            # Créer le nom de la vidéo temporaire (sans audio)
            temp_video = video_path.replace('.mp4', '_temp_no_audio.mp4')
            
            # Renommer la vidéo actuelle (sans audio) en temporaire
            if os.path.exists(video_path):
                os.rename(video_path, temp_video)
            else:
                print(f"⚠️  Fichier vidéo introuvable: {video_path}")
                return
            
            # Vérifier si la vidéo originale a de l'audio
            has_original_audio = False
            if original_video_path and os.path.exists(original_video_path):
                try:
                    # Vérifier avec ffprobe si la vidéo a une piste audio
                    result = subprocess.run(
                        ['ffprobe', '-v', 'error', '-select_streams', 'a:0', 
                         '-show_entries', 'stream=codec_type', '-of', 'csv=p=0', original_video_path],
                        capture_output=True, timeout=5
                    )
                    has_original_audio = result.returncode == 0 and result.stdout.strip() == b'audio'
                except:
                    has_original_audio = False
            
            # Utiliser ffmpeg pour combiner vidéo et audio
            # Vérifier si ffmpeg est disponible
            print("  → Vérification de ffmpeg...")
            try:
                result = subprocess.run(['ffmpeg', '-version'], 
                            capture_output=True, check=True, timeout=5)
                ffmpeg_available = True
                print("  ✅ ffmpeg détecté et prêt")
            except (subprocess.CalledProcessError, FileNotFoundError, subprocess.TimeoutExpired):
                ffmpeg_available = False
                print("  ⚠️  ffmpeg non trouvé, tentative avec moviepy...")
            
            if not ffmpeg_available:
                # Essayer avec moviepy
                video = None
                audio = None
                final = None
                try:
                    from moviepy.editor import VideoFileClip, AudioFileClip, CompositeVideoClip
                    
                    print("  → Utilisation de moviepy pour combiner vidéo et audio...")
                    
                    # Charger la vidéo et l'audio
                    video = VideoFileClip(temp_video)
                    audio = AudioFileClip(temp_audio)
                    
                    # Combiner - utiliser la méthode correcte selon la version de moviepy
                    # D'abord vérifier quelle méthode est disponible
                    if hasattr(video, 'with_audio'):
                        # Nouvelle version de moviepy
                        final = video.with_audio(audio)
                    elif hasattr(video, 'set_audio'):
                        # Ancienne version de moviepy
                        final = video.set_audio(audio)
                    else:
                        # Fallback avec CompositeVideoClip
                        final = CompositeVideoClip([video])
                        if hasattr(final, 'set_audio'):
                            final = final.set_audio(audio)
                        elif hasattr(final, 'with_audio'):
                            final = final.with_audio(audio)
                        else:
                            raise AttributeError("Aucune méthode pour ajouter l'audio trouvée dans moviepy")
                    
                    # Écrire la vidéo finale
                    final.write_videofile(
                        video_path, 
                        codec='libx264', 
                        audio_codec='aac', 
                        verbose=False, 
                        logger=None,
                        temp_audiofile=video_path.replace('.mp4', '_temp_audio.m4a')
                    )
                    
                    print(f"✅ Vidéo finale avec audio sauvegardée: {video_path}")
                    
                except ImportError as e:
                    print("\n" + "="*60)
                    print("❌ ERREUR: moviepy n'est pas installé")
                    print("="*60)
                    print("\n📦 OPTIONS D'INSTALLATION:")
                    print("\n1. Installer moviepy (recommandé si ffmpeg n'est pas disponible):")
                    print("   pip install moviepy")
                    print("\n2. OU installer ffmpeg (meilleure option, plus rapide):")
                    print("   Windows: Téléchargez depuis https://ffmpeg.org/download.html")
                    print("   Ou utilisez: winget install ffmpeg")
                    print("\n3. Si vous avez des problèmes avec moviepy, essayez:")
                    print("   pip install --upgrade moviepy")
                    print("   pip install imageio imageio-ffmpeg")
                    print("\n" + "="*60)
                    # Restaurer la vidéo originale
                    if os.path.exists(temp_video):
                        # Supprimer le fichier cible s'il a été partiellement créé
                        try:
                            if os.path.exists(video_path):
                                os.remove(video_path)
                        except:
                            pass
                        os.rename(temp_video, video_path)
                    return
                except Exception as e:
                    print(f"\n⚠️  Erreur lors de l'utilisation de moviepy: {e}")
                    print("   Essayez d'installer ffmpeg à la place (plus fiable)")
                    # Restaurer la vidéo originale
                    if os.path.exists(temp_video):
                        # Supprimer le fichier cible s'il a été partiellement créé
                        try:
                            if os.path.exists(video_path):
                                os.remove(video_path)
                        except:
                            pass
                        os.rename(temp_video, video_path)
                    return
                finally:
                    # Fermer TOUS les clips AVANT de manipuler les fichiers
                    if final is not None:
                        try:
                            final.close()
                        except:
                            pass
                    if audio is not None:
                        try:
                            audio.close()
                        except:
                            pass
                    if video is not None:
                        try:
                            video.close()
                        except:
                            pass
                    
                    # Attendre un peu pour que les handles de fichiers soient libérés
                    import time
                    time.sleep(0.5)
                    
                    # Nettoyer les fichiers temporaires
                    try:
                        if os.path.exists(temp_video):
                            os.remove(temp_video)
                    except:
                        pass
                    try:
                        if os.path.exists(temp_audio):
                            os.remove(temp_audio)
                    except:
                        pass
                    
                    # Si on a réussi (final is not None), on peut retourner
                    if final is not None:
                        return
            
            # Utiliser ffmpeg pour combiner vidéo et audio
            # Le fichier final sera directement output_annotated.mp4
            print("  → Utilisation de ffmpeg pour combiner vidéo et audio...")
            
            if has_original_audio:
                # Extraire l'audio original de la vidéo d'entrée
                print("  → Détection d'audio original, extraction...")
                temp_original_audio = video_path.replace('.mp4', '_original_audio_temp.wav')
                extract_cmd = [
                    'ffmpeg', '-i', original_video_path, '-vn', '-acodec', 'pcm_s16le',
                    '-ar', '44100', '-ac', '1', '-y', temp_original_audio
                ]
                try:
                    subprocess.run(extract_cmd, capture_output=True, check=True, timeout=60)
                    print("  → Audio original extrait, mixage avec l'alarme...")
                    
                    # Mixer l'audio original avec l'alarme
                    # Utiliser amix pour mixer les deux pistes audio
                    cmd = [
                        'ffmpeg', '-i', temp_video, '-i', temp_original_audio, '-i', temp_audio,
                        '-filter_complex', '[1:a][2:a]amix=inputs=2:duration=first:dropout_transition=2[a]',
                        '-c:v', 'copy', '-map', '0:v:0', '-map', '[a]', '-c:a', 'aac',
                        '-shortest', '-y', video_path
                    ]
                except Exception as e:
                    print(f"  ⚠️  Impossible d'extraire l'audio original ({e}), utilisation de l'alarme uniquement")
                    has_original_audio = False
            
            if not has_original_audio:
                # Utiliser seulement l'audio d'alarme
                cmd = [
                    'ffmpeg', '-i', temp_video, '-i', temp_audio,
                    '-c:v', 'copy', '-c:a', 'aac', '-map', '0:v:0', '-map', '1:a:0',
                    '-shortest', '-y', video_path  # Écrire directement dans output_annotated.mp4
                ]
            
            try:
                result = subprocess.run(cmd, capture_output=True, check=True, timeout=300)
            except subprocess.TimeoutExpired:
                print("  ⚠️  ffmpeg a pris trop de temps, annulation...")
                raise
            except subprocess.CalledProcessError as e:
                error_msg = e.stderr.decode('utf-8', errors='ignore') if e.stderr else str(e)
                print(f"  ⚠️  Erreur ffmpeg: {error_msg[:200]}")
                raise
            
            # Nettoyer les fichiers temporaires
            if os.path.exists(temp_video):
                os.remove(temp_video)
            if os.path.exists(temp_audio):
                os.remove(temp_audio)
            if 'temp_original_audio' in locals() and os.path.exists(temp_original_audio):
                os.remove(temp_original_audio)
            
            if has_original_audio:
                print(f"✅ Vidéo finale avec audio original + alarme sauvegardée: {video_path}")
            else:
                print(f"✅ Vidéo finale avec alarme sauvegardée: {video_path}")
            
        except Exception as e:
            print(f"⚠️  Erreur lors de l'ajout de l'audio: {e}")
            print("   La vidéo sans audio est disponible.")
            # Attendre un peu pour libérer les handles de fichiers
            import time
            time.sleep(2)
            # Restaurer la vidéo originale si elle existe
            if 'temp_video' in locals() and os.path.exists(temp_video):
                try:
                    os.rename(temp_video, video_path)
                except PermissionError:
                    # Si toujours verrouillé, copier au lieu de renommer
                    try:
                        import shutil
                        if os.path.exists(video_path):
                            try:
                                os.remove(video_path)
                            except:
                                pass
                        time.sleep(0.5)
                        shutil.copy2(temp_video, video_path)
                        time.sleep(0.5)
                        try:
                            os.remove(temp_video)
                        except:
                            pass
                    except Exception as e2:
                        print(f"   ⚠️  Impossible de restaurer la vidéo. Le fichier temporaire est: {temp_video}")
                        print(f"   Vous pouvez le renommer manuellement en: {video_path}")
    
    def _load_tracking_data(self, tracking_data_path):
        """
        Charge les données de tracking depuis un fichier CSV
        Format attendu: track_id, frame, x, y
        """
        tracking_data = defaultdict(list)
        
        with open(tracking_data_path, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                # Adaptation aux noms de colonnes (frame vs frame_id, x vs x_center)
                frame_key = 'frame_id' if 'frame_id' in row else 'frame'
                x_key = 'x_center' if 'x_center' in row else 'x'
                y_key = 'y_center' if 'y_center' in row else 'y'
                
                try:
                    frame = int(row[frame_key])
                    tracking_data[frame].append({
                        'track_id': int(row['track_id']),
                        'x': float(row[x_key]),
                        'y': float(row[y_key])
                    })
                except (ValueError, KeyError) as e:
                    print(f"Warning: Skipping invalid row: {row} ({e})")
                    continue
        
        return tracking_data
    
    def _prepare_visualization_data(self, result, current_video_frame):
        """Prépare les données pour la visualisation"""
        track_id = result['track_id']
        
        # Récupération de l'historique des positions pour la trajectoire
        # Utiliser toutes les positions jusqu'à maintenant pour affichage progressif
        trajectory = [(r['x'], r['y']) 
                     for r in self.results[track_id]]
        
        # Vitesse
        velocity = result['motion_features']['instant_velocity']
        
        return {
            'track_id': track_id,
            'x': result['x'],  # Position brute du CSV
            'y': result['y'],  # Position brute du CSV
            'x_smooth': result['x_smooth'],  # Garder aussi pour trajectoire si besoin
            'y_smooth': result['y_smooth'],
            'status': result['status'],
            'trajectory': trajectory,  # Toute la trajectoire jusqu'à maintenant
            'predicted_trajectory': result['predicted_trajectory'],
            'confidence': result['confidence'],
            'threat_score': result['threat_score'],
            'velocity': velocity,
            'behavior': result['behavior'],
            'frame': result['frame']  # Ajouter le numéro de frame de tracking
        }
    
    def export_results(self, output_dir):
        """
        Export des résultats en JSON et CSV
        
        Args:
            output_dir (str): répertoire de sortie
        """
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        # Export JSON (complet)
        json_path = output_path / "analysis_results.json"
        with open(json_path, 'w') as f:
            # Conversion des résultats pour JSON
            json_results = {}
            for track_id, data in self.results.items():
                json_results[str(track_id)] = [
                    {k: v for k, v in item.items() 
                     if not isinstance(v, np.ndarray)}
                    for item in data
                ]
            json.dump(json_results, f, indent=2, default=str)
        
        print(f"Résultats JSON exportés: {json_path}")
        
        # Export CSV (simplifié)
        csv_path = output_path / "analysis_summary.csv"
        with open(csv_path, 'w', newline='') as f:
            fieldnames = [
                'track_id', 'frame', 'x', 'y', 'status', 
                'threat_score', 'confidence', 'velocity_magnitude',
                'acceleration_magnitude', 'direction_change'
            ]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            
            for track_id, data in self.results.items():
                for item in data:
                    writer.writerow({
                        'track_id': track_id,
                        'frame': item['frame'],
                        'x': f"{item['x_smooth']:.2f}",
                        'y': f"{item['y_smooth']:.2f}",
                        'status': item['status'],
                        'threat_score': f"{item['threat_score']:.1f}",
                        'confidence': f"{item['confidence']:.3f}",
                        'velocity_magnitude': f"{item['motion_features']['instant_velocity']['magnitude']:.2f}",
                        'acceleration_magnitude': f"{item['motion_features']['acceleration']['magnitude']:.2f}",
                        'direction_change': f"{item['motion_features']['direction_change']:.2f}"
                    })
        
        print(f"Résumé CSV exporté: {csv_path}")
        
        # Statistiques globales
        self._export_statistics(output_path)
    
    def _export_statistics(self, output_path):
        """Export des statistiques globales"""
        stats_path = output_path / "statistics.txt"
        
        with open(stats_path, 'w') as f:
            f.write("=== STATISTIQUES D'ANALYSE ===\n\n")
            
            for track_id, data in self.results.items():
                analyzer = self.behavior_analyzers.get(track_id)
                if not analyzer:
                    continue
                
                summary = analyzer.get_behavior_summary()
                
                f.write(f"Drone ID {track_id}:\n")
                f.write(f"  - Frames analysées: {len(data)}\n")
                f.write(f"  - Score de menace moyen: {summary['average_threat_score']:.1f}\n")
                f.write(f"  - Score de menace max: {summary['max_threat_score']:.1f}\n")
                f.write(f"  - Comportement NORMAL: {summary['normal_count']} frames\n")
                f.write(f"  - Comportement SUSPECT: {summary['suspect_count']} frames\n")
                f.write(f"  - Comportement HOSTILE: {summary['hostile_count']} frames\n")
                f.write("\n")
        
        print(f"Statistiques exportées: {stats_path}")


def main():
    """Fonction principale"""
    # Configuration
    # Zone restreinte : définir les points du polygone rouge
    # Centre du carré rouge à la position x=518, y=663
    # Format: [(x1, y1), (x2, y2), (x3, y3), (x4, y4), ...]
    # Taille du carré : 250x250 pixels (125 pixels de chaque côté du centre)
    center_x, center_y = 518, 663
    square_size = 250  # Taille du carré en pixels (augmentée)
    half_size = square_size // 2
    
    restricted_zone = [
        (center_x - half_size, center_y - half_size),  # Point 1 - Coin supérieur gauche
        (center_x + half_size, center_y - half_size),  # Point 2 - Coin supérieur droit
        (center_x + half_size, center_y + half_size),  # Point 3 - Coin inférieur droit
        (center_x - half_size, center_y + half_size)   # Point 4 - Coin inférieur gauche
    ]
    
    config = {
        'trail_length': 30,
        'prediction_steps': 15,
        'restricted_zone': restricted_zone  # Zone restreinte
    }
    
    # Chemins
    # Chemins relatifs basés sur la structure du projet
    # Notebook est dans step3_trajectory_analysis/
    VIDEO_PATH = r"..\drone_videos\drone_vid1.mp4"
    TRACKING_DATA_PATH = r"..\step2_tracking\tracking_data.csv"
    OUTPUT_VIDEO_PATH = "output_annotated.mp4"
    OUTPUT_DIR = "results"
    
    print("🚁 Démarrage de l'analyse de trajectoire de drones")
    print("=" * 60)
    
    # Initialisation du système
    analyzer = DroneTrajectoryAnalyzer(config)
    
    # Traitement de la vidéo
    try:
        analyzer.process_video(VIDEO_PATH, TRACKING_DATA_PATH, OUTPUT_VIDEO_PATH)
        
        # Export des résultats
        analyzer.export_results(OUTPUT_DIR)
        
        # Génération de la visualisation matplotlib
        trajectory_plot_path = str(Path(OUTPUT_DIR) / "trajectory_plot.png")
        DroneVisualizer.plot_trajectory_matplotlib(
            analyzer.results, 
            output_path=trajectory_plot_path
        )
        
        print("\n✅ Analyse terminée avec succès!")
        print(f" Vidéo annotée: {OUTPUT_VIDEO_PATH}")
        print(f" Résultats: {OUTPUT_DIR}/")
        print(f" Visualisation trajectoire: {trajectory_plot_path}")
        
    except Exception as e:
        print(f"\n Erreur lors de l'analyse: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

🚁 Démarrage de l'analyse de trajectoire de drones
Traitement de la vidéo: 1601 frames vidéo, 1561 positions de tracking
Chaque position sera affichée pendant environ 1 frames
Progression: 30/1601 frames
Progression: 60/1601 frames
Progression: 90/1601 frames
Progression: 120/1601 frames
Progression: 150/1601 frames
Progression: 180/1601 frames
Progression: 210/1601 frames
Progression: 240/1601 frames
Progression: 270/1601 frames
Progression: 300/1601 frames
Progression: 330/1601 frames
Progression: 360/1601 frames
Progression: 390/1601 frames
Progression: 420/1601 frames
Progression: 450/1601 frames
Progression: 480/1601 frames
Progression: 510/1601 frames
Progression: 540/1601 frames
Progression: 570/1601 frames
Progression: 600/1601 frames
Progression: 630/1601 frames
Progression: 660/1601 frames
Progression: 690/1601 frames
Progression: 720/1601 frames
Progression: 750/1601 frames
Progression: 780/1601 frames
Progression: 810/1601 frames
Progression: 840/1601 frames
Progression: 870

## 8. Exécution

In [9]:
if __name__ == "__main__":
    # Définition de la zone restreinte (Carré central)
    center_x, center_y = 518, 663
    size = 250
    restricted_zone = [
        (center_x - size//2, center_y - size//2),
        (center_x + size//2, center_y - size//2),
        (center_x + size//2, center_y + size//2),
        (center_x - size//2, center_y + size//2)
    ]
    
    config = {
        'restricted_zone': restricted_zone,
        'prediction_steps': 15
    }
    
    analyzer = DroneTrajectoryAnalyzer(config)
    
    # Chemins (A ADAPTER)
    VIDEO_INPUT = r"C:\Studies\S7\AI\drone_detection_project\drone_videos\drone_vid1.mp4"
    CSV_INPUT = r"C:\Studies\S7\AI\drone_detection_project\step2_tracking\tracking_data.csv"
    VIDEO_OUTPUT = "output_notebook.mp4"
    
    if os.path.exists(VIDEO_INPUT) and os.path.exists(CSV_INPUT):
        analyzer.process_video(VIDEO_INPUT, CSV_INPUT, VIDEO_OUTPUT)
    else:
        print("⚠️ Fichiers d'entrée non trouvés. Vérifiez les chemins.")

Traitement de la vidéo: 1601 frames vidéo, 1561 positions de tracking
Chaque position sera affichée pendant environ 1 frames
Progression: 30/1601 frames
Progression: 60/1601 frames
Progression: 90/1601 frames
Progression: 120/1601 frames
Progression: 150/1601 frames
Progression: 180/1601 frames
Progression: 210/1601 frames
Progression: 240/1601 frames
Progression: 270/1601 frames
Progression: 300/1601 frames
Progression: 330/1601 frames
Progression: 360/1601 frames
Progression: 390/1601 frames
Progression: 420/1601 frames
Progression: 450/1601 frames
Progression: 480/1601 frames
Progression: 510/1601 frames
Progression: 540/1601 frames
Progression: 570/1601 frames
Progression: 600/1601 frames
Progression: 630/1601 frames
Progression: 660/1601 frames
Progression: 690/1601 frames
Progression: 720/1601 frames
Progression: 750/1601 frames
Progression: 780/1601 frames
Progression: 810/1601 frames
Progression: 840/1601 frames
Progression: 870/1601 frames
Progression: 900/1601 frames
Progress

## 📊 Behavioral Analysis Results Summary

Analysis performed on  across **7 tracked drones** (1,671 total frames):

| Drone ID | Frames | Avg Threat | Max Threat | Normal | Suspect | Hostile |
|----------|--------|-----------|------------|--------|---------|---------|
| 1 | 388 | 33.1 | 80.0 | 291 (75.0%) | 20 (5.2%) | 77 (19.8%) |
| 2 | 10 | 7.0 | 35.0 | 8 (80.0%) | 2 (20.0%) | 0 (0.0%) |
| 5 | 6 | 21.7 | 60.0 | 3 (50.0%) | 2 (33.3%) | 1 (16.7%) |
| 13 | 5 | 7.0 | 35.0 | 4 (80.0%) | 1 (20.0%) | 0 (0.0%) |
| 16 | 7 | 0.0 | 0.0 | 7 (100%) | 0 (0.0%) | 0 (0.0%) |
| 21 | 13 | 25.4 | 60.0 | 10 (76.9%) | 2 (15.4%) | 1 (7.7%) |
| 22 | 1,242 | 26.4 | 80.0 | 1,183 (95.3%) | 22 (1.8%) | 37 (3.0%) |
| **Total** | **1,671** | — | — | **1,506 (90.1%)** | **49 (2.9%)** | **116 (6.9%)** |

### Threat Classification Rules
| Behavior | Threat Score | Criteria |
|----------|-------------|----------|
| Normal | 0–24 | Consistent speed/direction, no zone violations |
| Suspect | 25–59 | Sudden turns, moderate speed changes, near restricted zones |
| Hostile | 60–100 | High acceleration, erratic patterns, restricted zone penetration |

### Output Files
-  (40 MB) — annotated video with trajectories and threat alerts
-  — per-drone behavior breakdown
-  — frame-level classifications
-  — human-readable summary
-  — trajectory visualization